# Fraud Detection in Electricity and Gas Consumption Challenge

This is a simple starter notebook based on the tutorial prepared by Joy Wawira. Check out the article [here](https://zindi.africa/learn/fraud-detection-in-electricity-and-gas-consumption-challenge-tutorial) for a more detailed description of the steps taken.

This notebook covers:
- Downloading the data straight from Zindi and onto colab
- Loading the data and carrying out simple EDA to understand the data and prepare for modelling 
- Preprocessing the data and feature engineering 
- Creating a simple LGBM model and predicting on the test set
- Prepare submission file and save as csv
- Some tips on how to improve model performance and your score

# Pre-Requisites

In [ ]:
DATA_DIR = '../data'

TRAIN_DIR = f'{DATA_DIR}/train'
TEST_DIR = f'{DATA_DIR}/test'

OUTPUT_DIR = f'{DATA_DIR}/output'

In [ ]:
import os

for pth in [TRAIN_DIR, TEST_DIR, OUTPUT_DIR]:
    if not os.path.exists(pth):
        os.makedirs(pth)


# Data Prep

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm

from sklearn.metrics.pairwise import rbf_kernel

from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline, make_pipeline, FunctionTransformer
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn import metrics
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImPipeline
from datetime import date

from catboost import CatBoostClassifier

import optuna
from sklearn.metrics import accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
import mlflow
import mlflow.sklearn
import warnings
warnings.simplefilter('ignore')

## Read the Data

In [ ]:
client_train = pd.read_csv(f'{TRAIN_DIR}/client_train.csv', low_memory=False)
invoice_train = pd.read_csv(f'{TRAIN_DIR}/invoice_train.csv', low_memory=False)

client_test = pd.read_csv(f'{TEST_DIR}/client_test.csv', low_memory=False)
invoice_test = pd.read_csv(f'{TEST_DIR}/invoice_test.csv', low_memory=False)
sample_submission = pd.read_csv(f'{DATA_DIR}/SampleSubmission.csv', low_memory=False)

In [ ]:
import logging

# Setup basic configuration for logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')



## Data Understanding

In [ ]:
#compare size of the various datasets
print(client_train.shape, invoice_train.shape, client_test.shape, invoice_train.shape)

In [ ]:
#print top rows of dataset
invoice_train.head()

In [ ]:
#print top rows of dataset
client_train.head()

In [ ]:
#Get a summary for all numerical columns
invoice_train.describe()

In [ ]:
#Get a summary for all numerical columns
client_train.describe()

In [ ]:
#Get concise information of each column in dataset
invoice_train.info()

In [ ]:
invoice_train['counter_coefficient'].value_counts()

In [ ]:
#Get concise information of each column in dataset
client_train.info()

In [ ]:
#Getting unique values on the invoice train data
for col in invoice_train.columns:
    print(f"{col} - {invoice_train[col].nunique()}")

In [ ]:
#Getting unique values on the invoice train data
for col in client_train.columns:
    print(f"{col} - {client_train[col].nunique()}")

In [ ]:
#check for missing values
invoice_train.isnull().sum()

In [ ]:
#check for missing values
client_train.isnull().sum()

No missing values in train set

In [ ]:
#Visualize fraudulent activities
fraudactivities = client_train.groupby(['target'])['client_id'].count()
plt.bar(x=fraudactivities.index, height=fraudactivities.values, tick_label = [0,1])
plt.title('Fraud - Target Distribution')
plt.show()

Target is highly imbalanced with fewer cases of fraudulent activities

In [ ]:
fraudactivities

In [ ]:
#Visualize client distribution across districts and regions
for col in ['disrict','region']:
    region = client_train.groupby([col])['client_id'].count()
    plt.bar(x=region.index, height=region.values)
    plt.title(col+' distribution')
    plt.show()

In [ ]:
# generate graph for rbf kernel

ages = np.linspace(client_train[["region"]].min()-20,
                   client_train[["region"]].max(),
                   500).reshape(-1, 1)
gamma1 = 0.005

rbf1 = rbf_kernel(ages, [[105]], gamma=gamma1)


fig, ax1 = plt.subplots()

ax1.set_xlabel("Housing median age")
ax1.set_ylabel("Number of districts")
ax1.hist(client_train[["region"]], bins=50)

ax2 = ax1.twinx()  # create a twin axis that shares the same x-axis
color = "blue"
ax2.plot(ages, rbf1, color=color, label="gamma = 0.005")

ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylabel("Region similarity", color=color)

plt.legend(loc="upper left")

plt.show()

## Feature Engineering

In [ ]:
#convert the column invoice_date to date time format on both the invoice train and invoice test
for df in [invoice_train,invoice_test]:
    df['invoice_date'] = pd.to_datetime(df['invoice_date'])

In [ ]:
target = client_train['target']
client_train = client_train.drop(columns=['target'])

In [ ]:
# This function aggregates the invoice data by client ID and merges it with the client data.
# It calculates various statistics for different columns in the invoice data.
# The input parameters are:
# - client_data: The client data dataframe.
# - invoice_data: The invoice data dataframe.
# The function returns the merged dataframe.

# feature on Invoice does not come in consistent pattern (3 invoices have back to back)
# fruad people mess up counter/index --> eng features how much energy consumed since last time 

# consumption level (infrench) add all 4 levels 


#index is the number on the counter 

#if you substract these two and substract features (something is going on) --> take a difference btw them

# old index new index and total consumption level


# how many days elapse since last invoice; how much energy data they use on average 
# how many unique counter they have (17)

# some values are way too large (some column just moved, copy paste); move these back 


# some test ID exist both in training and test set; --> email stephen 

def aggregate_by_client_id(client_data, invoice_data):
        #feature total consumption levels vs index difference
        invoice_data['consumption_diff'] = invoice_data[['consommation_level_1', 'consommation_level_2','consommation_level_3','consommation_level_4']].sum(axis=1) 
        -( invoice_data['new_index'] - invoice_data['old_index'])
        
        aggs = {}
        aggs['invoice_date'] = ['min','max','count']
        aggs['consumption_diff'] = ['sum','mean']
        aggs['tarif_type'] = ['count']
        aggs['reading_remarque'] = ['mean']
        aggs['counter_statue'] = ['count']
        aggs['counter_coefficient'] = ['mean']
        aggs['consommation_level_1'] = ['mean','sum']
        aggs['consommation_level_2'] = ['mean','sum']
        aggs['consommation_level_3'] = ['mean','sum']
        aggs['consommation_level_4'] = ['mean','sum']
        # how many unique counters do they have
        aggs['counter_number'] = ['nunique']
        aggs['months_number'] = ['mean']
        
        # Group the invoice data by client ID and calculate the aggregated statistics
        invoice_agg = invoice_data.groupby(['client_id']).agg(aggs)
        
        # Rename the columns of the aggregated dataframe
        invoice_agg.columns = ['_'.join(col).strip() for col in invoice_agg.columns.values]
        
        # Reset the index of the aggregated dataframe
        invoice_agg.reset_index(inplace=True)
        
        #add two features Frequency of the invoice;the energy consumption on average per day
        invoice_agg[['invoice_date_max','invoice_date_min']] = invoice_agg[['invoice_date_max','invoice_date_min']].apply(pd.to_datetime)
        invoice_agg['tenure_days']= (invoice_agg['invoice_date_max'] - invoice_agg['invoice_date_min']).dt.days

        invoice_agg['invoice_freq'] = invoice_agg['tenure_days']/invoice_agg['invoice_date_count']
        invoice_agg['consumption_sum'] = invoice_agg[['consommation_level_1_sum', 'consommation_level_2_sum','consommation_level_3_sum']].sum(axis=1)

        #if tenure days is 0, keep it as it is, else divide by tenure days
        invoice_agg['daily_consumption'] = invoice_agg.apply(lambda x: x['consumption_sum'] if x['tenure_days'] == 0 else x['consumption_sum']/x['tenure_days'], axis=1)

        columns_to_drop = ['invoice_date_min', 'invoice_date_max', 'invoice_date_count', 'consommation_level_1_sum', 'consommation_level_2_sum', 'consommation_level_3_sum', 
        'consommation_level_4_sum', 'consumption_sum']
        invoice_agg.drop(columns_to_drop, axis=1, inplace=True)

        client_data['creation_date'] = pd.to_datetime(client_data['creation_date'])
        client_data['creation_day_diff'] = (client_data['creation_date'] - pd.Timestamp(date.today())).dt.days.astype(int)


        # Merge the aggregated invoice data with the client data
        invoice_final = pd.merge(client_data, invoice_agg, on='client_id', how='left')

        #drop two remaining cat features client_id column and creation_date
        invoice_final.drop(['client_id', 'creation_date'], axis=1, inplace=True)
        
        # Return the merged dataframe
        return invoice_final


In [ ]:
agg_train = aggregate_by_client_id(client_train , invoice_train)
agg_test = aggregate_by_client_id(client_test , invoice_test) 

In [ ]:
# create multiple features based on region values
def rbf_transformer(Y):
    return make_pipeline(
        FunctionTransformer(rbf_kernel, kw_args=dict(Y=Y, gamma=0.1),feature_names_out="one-to-one"),
        StandardScaler()
    )
# def num_transformer():
#     return make_pipeline(
#             FunctionTransformer(lambda x: x.astype(int), validate=False)
#         )




preprocessing = ColumnTransformer([
 ('onehot', OneHotEncoder(handle_unknown="ignore"), ['disrict']),
 ("rbf_transformer_105", rbf_transformer([[105]]), ["region"]),
 ("rbf_transformer_305", rbf_transformer([[305]]), ["region"]),
 ("rbf_transformer_375", rbf_transformer([[375]]), ["region"]),
 ], remainder="passthrough")

# client_train_preped = preprocessing.fit_transform(agg_train)

# preprocessing.get_feature_names_out()

# Modelling

In [ ]:
logging.info("Training Models...")

In [ ]:
#potential to change the k neighbours; sampling strategy 

light_GBM =ImPipeline([
    ('preprocessing', preprocessing),
    # ('smote', SMOTE(sampling_strategy = 0.3,random_state=42)),
    
    ('model', LGBMClassifier(boosting_type='gbdt',class_weight='balanced',))
])
light_GBM.fit(agg_train, target)

In [ ]:


fraud_predictions = light_GBM.predict_proba(agg_test)


fraud_predictions = fraud_predictions[:, 1]

## rf tuning

In [ ]:


def objective(trial,X,y):
    # Define the hyperparameters to be optimized
    max_samples = trial.suggest_float("max_samples", 0.5, 0.7)  # Fraction of samples to be used for each tree
    min_samples_split = trial.suggest_int("min_samples_split", 2, 100)

    # Create the pipeline with the suggested parameters
    clf_rf = ImPipeline([
        ('preprocessing', preprocessing),
        # ('smote', SMOTE(sampling_strategy=0.3, random_state=42)),  # Adjust as needed
        ('model', RandomForestClassifier(n_estimators=1000,
                                             max_samples=max_samples, 
                                             min_samples_split=min_samples_split, 
                                             class_weight= 'balanced',
                                             n_jobs=3,
                                             random_state=42))
    ])
    
    # # Fit the model
    # clf_rf.fit(agg_train, target)
    cv_scores = cross_val_score(clf_rf, X, y, cv=5, n_jobs=3, scoring="roc_auc")
    score = np.mean(cv_scores)
    # scorestd = cv_scores.std()
    return score  # Replace with appropriate metric

# Create a study object and specify the direction is 'maximize'.
study = optuna.create_study(direction='maximize')

# Start the optimization
study.optimize(lambda trial: objective(trial, agg_train, target), n_trials=15,  gc_after_trial=True)

# Print the optimal parameters
print(study.best_params)


In [ ]:
study.best_params

In [ ]:
clf_rf = ImPipeline([
        ('preprocessing', preprocessing),
        # ('smote', SMOTE(sampling_strategy=0.3, random_state=42)),  # Adjust as needed
        ('model', RandomForestClassifier(n_estimators=1000,
                                            #  **study.best_params,
                                            max_samples= 0.605199046299782, min_samples_split= 60,
                                            class_weight='balanced',
                                             n_jobs=-1,
                                             random_state=42))
    ])

cv_scores = cross_val_score(clf_rf, agg_train, target, cv=5, n_jobs=-1, scoring="roc_auc")
# clf_rf.fit(agg_train, target)
# fraud_predictions = clf_rf.predict_proba(agg_test)


# fraud_predictions = fraud_predictions[:, 1]


In [ ]:
# cv_scores.mean()

## Tune catboost

In [ ]:
def objective(trial,X,y):
    param = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 4, 10),  # CatBoost uses 'depth' instead of 'num_leaves'
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'max_bin': trial.suggest_int('max_bin', 100, 300),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0, 5),  # Similar to 'lambda_l1' and 'lambda_l2'
        # Fixed parameters
        'iterations': 1000,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'auto_class_weights': 'Balanced',
        'thread_count':4,
        # Add other parameters as needed
    }
    cat_boost = ImPipeline([
            ('preprocessing', preprocessing),
            # ('smote', SMOTE(sampling_strategy=0.3, random_state=42)),  # Adjust as needed
            ('model', CatBoostClassifier(**param))
        ])

    cv_scores = cross_val_score(cat_boost, X, y, cv=5, n_jobs=-1, scoring="roc_auc")
    score = np.mean(cv_scores)
    return score

# Create a study object and specify the direction is 'maximize'.
study = optuna.create_study(direction='maximize')

# Start the optimization
study.optimize(lambda trial: objective(trial, agg_train, target), n_trials=100,  gc_after_trial=True)

# Print the optimal parameters
print(study.best_params)

In [ ]:
study.best_params
#best cv score is 0.8363991169354101

In [ ]:
# Example parameter values
param = {
    'learning_rate': 0.1,  # Example fixed value
    'max_depth': 6,            # Example fixed value
    'min_data_in_leaf': 20,  # Example fixed value
    'max_bin': 100,
    'l2_leaf_reg': 3.0,    # Example fixed value
    # You can add other CatBoost specific parameters here
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'auto_class_weights': 'Balanced',
    'iterations': 1000,
    'thread_count':4,
}


# Example of using these parameters in CatBoostClassifier




cat_boost = ImPipeline([
        ('preprocessing', preprocessing),
        # ('smote', SMOTE(sampling_strategy=0.3, random_state=42)),  # Adjust as needed
        ('model', CatBoostClassifier(**param))
    ])

cv_scores = cross_val_score(cat_boost, agg_train, target, cv=5, n_jobs=-1, scoring="roc_auc")

In [ ]:
cv_scores

In [ ]:
gradient_GBM =ImPipeline([
    ('preprocessing', preprocessing),
    ('smote', SMOTE(sampling_strategy = 0.3,random_state=42)),
    ('model', GradientBoostingClassifier())
])
gradient_GBM.fit(agg_train, target)
fraud_predictions = gradient_GBM.predict(agg_test)

In [ ]:
xgboost_pipeline = ImPipeline([
    ('preprocessing', preprocessing),
    ('smote', SMOTE(sampling_strategy=0.3, random_state=42)),
    ('model', xgb.XGBClassifier())
])

xgboost_pipeline.fit(agg_train, target)
fraud_predictions = xgboost_pipeline.predict(agg_test)


In [ ]:
preds = pd.DataFrame(fraud_predictions, columns=['target'])
preds.head()
sub_client_id = client_test['client_id']
submission = pd.DataFrame(
    {
        'client_id': sub_client_id,
        'target': preds['target']
    }
)

submission.head()

In [ ]:
submission.to_csv(f'{OUTPUT_DIR}/submission_lightbgm_new_features.csv', index=False)

## Train LGBM *Classifier* 

### the following code is to log MLflow

In [ ]:
# X = train.drop(columns=['target'])
# y = train['target']

### Train Test split

In [ ]:
# # Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Create a pipeline
LGBM_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LGBMClassifier(boosting_type='gbdt'))
])
mlflow_run_tracking('LGBMClassifier',LGBM_pipeline, X_train, y_train, X_test, y_test )

In [ ]:
# Create a pipeline
decision_tree_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier())
])


In [ ]:
# Create a pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC())
])
mlflow_run_tracking('SVM',svm_pipeline, X_train, y_train, X_test, y_test )

In [ ]:
def mlflow_run_tracking(model_name, model, X_train, y_train, X_test, y_test):
    """
    Function to track model training using MLflow.

    Parameters:
    - model_name (str): Name of the model.
    - model: The machine learning model object.
    - X_train: Training data features.
    - y_train: Training data labels.
    - X_test: Testing data features.
    - y_test: Testing data labels.

    Example:
    mlflow_run_tracking("LGBMClassifier", LGBM_pipeline, X_train, y_train, X_test, y_test)
    """
    mlflow.set_tracking_uri('http://localhost:5000') 
    with mlflow.start_run():
        mlflow.set_tag("model", model_name)

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        AUC = metrics.roc_auc_score(y_test, predictions)

        mlflow.sklearn.log_model(LGBM_pipeline, model_name)


In [ ]:
mlflow.set_tracking_uri('http://localhost:5000') 
with mlflow.start_run():
    # Random Forest Regressor
    mlflow.set_tag("model", "LGBMClassifier")
    # grid_search_rf = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring='neg_mean_squared_error')
    # grid_search_rf.fit(X_train, y_train)
    
    LGBM_pipeline.fit(X_train, y_train)
    
    # best_rf = grid_search_rf.best_estimator_
    # mlflow.log_params(grid_search_rf.best_params_)

    # Fit the StandardScaler
    # decision_tree_pipeline['scaler'].fit(X_train)
    predictions = LGBM_pipeline.predict(X_test)
    AUC = metrics.roc_auc_score(y_test, predictions)

    mlflow.sklearn.log_model(LGBM_pipeline, "LGBMClassifier")


In [ ]:
mlflow.set_tracking_uri('http://localhost:5000')  # Replace with your MLflow tracking server URI

with mlflow.start_run():
    # Random Forest Regressor
    mlflow.set_tag("model", "DecisionTreeClassifier")
    # grid_search_rf = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring='neg_mean_squared_error')
    decision_tree_pipeline.fit(X_train, y_train)
    
    # best_rf = grid_search_rf.best_estimator_
    # mlflow.log_params(grid_search_rf.best_params_)

    # Fit the StandardScaler
    # decision_tree_pipeline['scaler'].fit(X_train)
    predictions = decision_tree_pipeline.predict(X_test)
    AUC = metrics.roc_auc_score(y_test, predictions)
    mlflow.log_metric("AUC", AUC)

    mlflow.sklearn.log_model(decision_tree_pipeline, "DecisionTreeClassifier")


In [ ]:
model = LGBMClassifier(boosting_type='gbdt', num_iterationss=500)
model.fit(x_train, y_train)

In [ ]:
x_train.disrict.unique()

## Make Predictions on test set

In [ ]:
test.columns

In [ ]:
preds = model.predict(test)
preds = pd.DataFrame(preds, columns=['target'])
preds.head()

In [ ]:
submission = pd.DataFrame(
    {
        'client_id': sub_client_id,
        'target': preds['target']
    }
)

submission.head()

In [ ]:
submission.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)

## Tips 
- Thorough EDA and incorporating domain knowledge
- Re-grouping categorical features
- More feature engineering(try utilizing some date-time features)
- Target balancing - oversampling, undersampling, SMOTE, scale_pos_weight
- Model ensembling
- Train-test split or cross-validation
